In [12]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

load_dotenv()
hf_token = os.environ["HUGGINGFACE_HUB_TOKEN"]
login(token=hf_token)
print(len(hf_token) != 0)

True


In [13]:
from transformers import AutoTokenizer

model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [14]:
import json
with open("chunks.json", "r") as f:
    summarized_chunks = json.load(f)

In [15]:
keys = list(summarized_chunks.keys())

In [16]:
from sentence_transformers import SentenceTransformer
from numpy import ndarray

transformer = SentenceTransformer("all-MiniLM-L6-v2")

In [17]:
embedded_keys = transformer.encode(keys)

In [18]:
import json
import faiss
import numpy as np

# Assume embeddings is a 2D numpy array of shape (num_chunks, dim)
dim = embedded_keys.shape[1]
index = faiss.IndexFlatL2(dim)  # using a simple L2 index
index.add(np.array(embedded_keys))  # add all chunk vectors

def to_text(index):
    key = keys[index]
    chunk = summarized_chunks.get(key)
    return "\n".join(chunk)
    
def rag_query(query):
    query_embedding = transformer.encode([query]) 
    k = 25
    distances, indicesList = index.search(query_embedding, k)
    indicesList = indicesList.tolist()
    return [to_text(index) for indices in indicesList for index in indices]

In [19]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda")
finetuned_model = finetuned_model = PeftModel.from_pretrained(
    base_model,
    "llama318-finetuned"
).to("cuda")
base_model.eval()
finetuned_model.eval()
print("Done") # just to avoid the .eval() print

Loading checkpoint shards: 100%|██████████| 4/4 [00:11<00:00,  2.89s/it]
[inference-ai ERROR (pid:5758 thread=124930027521088 allocator.c:53)]: Device 0 OOM 51529385984 / 51527024640
[inference-ai ERROR (pid:5758 thread=124930027521088 allocator.c:121)]: cuMemoryAllocate failed res=2
[inference-ai ERROR (pid:5758 thread=124930027521088 allocator.c:121)]: cuMemoryAllocate failed res=2


OutOfMemoryError: CUDA out of memory. Tried to allocate 224.00 MiB. GPU 0 has a total capacity of 47.99 GiB of which 751.87 MiB is free. Including non-PyTorch memory, this process has 47.30 GiB memory in use. Of the allocated memory 46.87 GiB is allocated by PyTorch, and 121.87 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
system_prompt = """
# Context
You are an expert in dungeons and dragons. You will be given excerpts from a transcript of a previous dungeons and dragons session
Not all excerpts may be relevant, you must decide which are most relevant and discard the rest
Excerpts will be delimited by === and a newline

# Objective
Answer the user’s question using **only**:
- The provided excerpts
- General Dungeons & Dragons rules or mechanics when needed for clarification

Expand on the response by describing any rules, spells, or mechanics mentioned in the answer

Do not introduce events, facts, or interpretations not supported by the excerpts.
If the answer is not in the excerpts, say 'Not specified in the session'

# Style
Write a short succinct answer.

# Tone
Neutral and matter-of-fact.

# Audience
Someone who participated in or watched the session, and is inquiring about specifics of the session

# Response format
Respond in a few short sentences, with clarifications on rules or mechanics where necessary.
"""

In [ ]:
def generate(conversation, model):
    device = model.device
    
    inputs = tokenizer.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device)
    
    input_length = len(inputs[0])
    outputs = model.generate(
        inputs=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=200,
        pad_token_id = tokenizer.eos_token_id
    )
    generated_tokens = outputs[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

def chat(query, model):
    excerpts = rag_query(query)
    conversation = [{
        "role": "system",
        "content": system_prompt
    }, {
        "role": "system",
        "content": "Excerpts:\n" + "\n===\n".join(excerpts)
    }, {
        "role": "user",
        "content": query
    }]
    return generate(conversation, model)

def rule():
    print("\n========================================\n")

def compare(query, verbose=False):
    print(query)
    if verbose:
        print("\n===\n".join(rag_query(query)))
    rule()
    print("Base Model:")
    print(chat(query, base_model))
    rule()
    print("Finetuned Model:")
    print(chat(query, finetuned_model))
    

In [11]:
compare("What did the party request Gustavo to cast on them")


What did the party request Gustavo to cast on them


Base Model:
They asked Gustavo to cast Speak with Animals on them.


Finetuned Model:
Gustavo was asked to cast Speak with Animals on the party.


In [34]:
compare("What did Gustavo request from the party")

What did Gustavo request from the party


Base Model:
Gustavo requested ten gold.


Finetuned Model:
Gustavo requested ten gold pieces, which the party gave him after he ritual cast a spell.


In [9]:
compare("Who did the party find to cast speak with animals on them")

NameError: name 'compare' is not defined

In [38]:
compare("What did Jack request from Tim while negotiating their contract")

What did Jack request from Tim while negotiating their contract


Base Model:
Jack asked Tim if he could cast Speak With Animals on him for research purposes, offering a gold piece in exchange.


Finetuned Model:
Jack asked Tim to cast Speak With Animals on him, offering one gold for the spell slot, to better communicate with the bird they desired to speak to.
